# Privacy Patents and CPC Code Trends (Loaded from sample_firm_privacy_patents.csv)

This notebook begins by loading the privacy patent dataset directly from `patents/sample_firm_privacy_patents.csv`.


## Step 1. Import packages

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


## Step 2. Define file locations

Load data directly from:
- `patents/sample_firm_privacy_patents.csv` (patent-level dataset)
- `privacy_cpc/privacy_cpc_codes_clustered_by_claude.csv` (72 CPC codes with definitions and Claude clusters)

In [ ]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_FOLDER = NOTEBOOK_DIR if (NOTEBOOK_DIR / "T4 - Toy Example [Topic Relevance on Privacy Innovation]").exists() else NOTEBOOK_DIR.parents[1]
SECTION_FOLDER = PROJECT_FOLDER / "T4 - Toy Example [Topic Relevance on Privacy Innovation]"
PATENTS_CSV = SECTION_FOLDER / "patents/sample_firm_privacy_patents.csv"
CLUSTERED_CSV = SECTION_FOLDER / "privacy_cpc/privacy_cpc_codes_clustered_by_claude.csv"

print(f"Patents file:   {PATENTS_CSV}")
print(f"Clustered file: {CLUSTERED_CSV}")
assert PATENTS_CSV.exists() and CLUSTERED_CSV.exists()

## Step 3. Load privacy patents dataset

Loads `sample_firm_privacy_patents.csv` and parses `filing_date` into `filing_year`.

In [ ]:
df_patents = pd.read_csv(PATENTS_CSV)
df_patents["filing_date"] = pd.to_datetime(df_patents["filing_date"])
df_patents["filing_year"] = df_patents["filing_date"].dt.year

print(f"Total patent-code rows: {len(df_patents):,}")
print(f"Unique patents: {df_patents['patent_num'].nunique():,}")
print(f"Unique firms (gvkey): {df_patents['gvkey'].nunique():,}")
print(f"Filing years: {df_patents['filing_year'].min()} to {df_patents['filing_year'].max()}")
display(df_patents.head(5))

## Step 4. Merge clustered CPC codes and assign cluster variables

Load `privacy_cpc_codes_clustered_by_claude.csv` and merge with `df_patents` on `privacy_cpc`.
Then, assign cluster flags using an explicit `if/elif/else` matching logic.

In [ ]:
# Load clustered CPC table
df_clustered = pd.read_csv(CLUSTERED_CSV)
df_clustered["cpc"] = df_clustered["cpc_display"].str.replace(" ", "", regex=False)

# Merge definition and original cluster group
df = df_patents.merge(
    df_clustered[["cpc", "cpc_display", "cpc_class", "definition", "group"]],
    left_on="privacy_cpc",
    right_on="cpc",
    how="left"
)

# Map clusters using if/then matching syntax
group_A_codes = df_clustered.loc[df_clustered["group"].str.startswith("A"), "cpc"].tolist()
group_B_codes = df_clustered.loc[df_clustered["group"].str.startswith("B"), "cpc"].tolist()
group_C_codes = df_clustered.loc[df_clustered["group"].str.startswith("C"), "cpc"].tolist()
group_D_codes = df_clustered.loc[df_clustered["group"].str.startswith("D"), "cpc"].tolist()

def assign_privacy_cluster(code):
    if code in group_A_codes:
        return "A. Privacy Protection (PII, anonymity, crypto)"
    elif code in group_B_codes:
        return "B. Use of Personal Data (tracking, profiling, targeting)"
    elif code in group_C_codes:
        return "C. Interception by Authorities"
    elif code in group_D_codes:
        return "D. Security (encryption, access control, storage)"
    else:
        return "Other"

df["privacy_cluster"] = df["privacy_cpc"].apply(assign_privacy_cluster)

print("Counts by Privacy Cluster (patent-code matches):")
display(df["privacy_cluster"].value_counts().to_frame("count"))

## Step 5. Create variables for the 4 focal privacy issues

We define explicit boolean flags for filtering and analysis:
1. **Mobile Telephony Protection**: Encryption, anonymity, and security in mobile/wireless communications (`H04W`)
2. **Online Ads, Cookies & Web Traffic**: Commercial profiling, user history, online advertisements, and cookie tracking (`G06Q30` and `G06F21/6263`)
3. **Anonymizing Data & Protecting/Hiding Identity**: Personal data de-identification, aliases, pseudonyms, anonymizers, and blind credentials
4. **Tracking User Activity (H04L)**: Network activity monitoring and user tracking over communication networks (`H04L67/535`, `H04L67/1396`)

In [ ]:
# 1. Mobile telephony protection (H04W)
mobile_codes = set(df_clustered.loc[df_clustered["cpc"].str.startswith("H04W"), "cpc"])

# 2. Online advertisements, cookies, and web traffic
online_ads_codes = set(df_clustered.loc[
    df_clustered["cpc"].str.startswith("G06Q30") | df_clustered["cpc"].eq("G06F21/6263"), 
    "cpc"
])

# 3. Anonymizing data, protecting and hiding identity
anonymizing_codes = set(df_clustered.loc[
    df_clustered["definition"].str.contains("anonym|hidden|hide|hiding|pseudonym|alias", case=False) |
    df_clustered["cpc"].isin(["G06F21/6245", "G06F21/6254", "H04W12/02", "H04W12/75", "G06Q20/383", "G06Q20/385"]),
    "cpc"
])

# 4. Tracking user activity (H04L)
tracking_h04l_codes = set(df_clustered.loc[
    df_clustered["cpc"].isin(["H04L67/535", "H04L67/1396", "H04L63/30"]) |
    (df_clustered["cpc"].str.startswith("H04L") & df_clustered["definition"].str.contains("tracking|monitoring.*activity", case=False)),
    "cpc"
])

# Matching logic functions
def match_issues(code):
    return pd.Series({
        "issue_1_mobile_telephony": code in mobile_codes,
        "issue_2_online_ads_cookies": code in online_ads_codes,
        "issue_3_anonymizing_identity": code in anonymizing_codes,
        "issue_4_tracking_h04l": code in tracking_h04l_codes,
    })

issue_flags = df["privacy_cpc"].apply(match_issues)
df = pd.concat([df, issue_flags], axis=1)

print("=== Summary of Focal Issues ===")
issues_summary = pd.DataFrame({
    "Patent-Code Matches": [
        df["issue_1_mobile_telephony"].sum(),
        df["issue_2_online_ads_cookies"].sum(),
        df["issue_3_anonymizing_identity"].sum(),
        df["issue_4_tracking_h04l"].sum()
    ],
    "Unique Patents": [
        df.loc[df["issue_1_mobile_telephony"], "patent_num"].nunique(),
        df.loc[df["issue_2_online_ads_cookies"], "patent_num"].nunique(),
        df.loc[df["issue_3_anonymizing_identity"], "patent_num"].nunique(),
        df.loc[df["issue_4_tracking_h04l"], "patent_num"].nunique()
    ],
    "Unique Firms": [
        df.loc[df["issue_1_mobile_telephony"], "gvkey"].nunique(),
        df.loc[df["issue_2_online_ads_cookies"], "gvkey"].nunique(),
        df.loc[df["issue_3_anonymizing_identity"], "gvkey"].nunique(),
        df.loc[df["issue_4_tracking_h04l"], "gvkey"].nunique()
    ]
}, index=[
    "1. Mobile telephony protection (H04W)",
    "2. Online ads, cookies, and web traffic",
    "3. Anonymizing data & hiding identity",
    "4. Tracking user activity (H04L)"
])
display(issues_summary)

## Step 6. Figure: Number of Patents by Group Over Filing Years (Seaborn Plot)

Plots the yearly trend of distinct patent counts for each CPC cluster group.

In [ ]:
# Aggregate unique patents by group and filing year
df["group_short"] = df["privacy_cluster"].apply(lambda s: s.split("(")[0].strip() if pd.notna(s) else s)
yearly_group = df.groupby(["group_short", "filing_year"])["patent_num"].nunique().reset_index()
yearly_group.rename(columns={"patent_num": "Patent Count", "filing_year": "Filing Year", "group_short": "Cluster Group"}, inplace=True)

plt.figure(figsize=(11, 6))
ax = sns.lineplot(
    data=yearly_group,
    x="Filing Year",
    y="Patent Count",
    hue="Cluster Group",
    marker="o",
    linewidth=2.2,
    palette="tab10"
)
plt.title("Number of Privacy Patents by CPC Cluster Group Over Time (1999–2021)", fontsize=14, weight="bold", pad=15)
plt.xlabel("Filing Year", fontsize=12)
plt.ylabel("Number of Distinct Patents Filed", fontsize=12)
plt.xticks(sorted(yearly_group["Filing Year"].unique())[::2], rotation=0)
plt.legend(title="Cluster Group", frameon=True, facecolor="white", framealpha=0.9)
plt.tight_layout()
plt.show()

## Step 7. Figure: Top 10 Assignees by Patent Count in Each Group (Horizontal Stacked Bar)

Identifies the top 10 firms by total privacy patents and plots horizontal stacked bars showing each firm's patent counts across the CPC cluster groups.

In [ ]:
# Clean firm names for title-case display
df["firm_display"] = df["conm"].str.title().str.replace(r"\s+-Cl\s+\w+$", "", regex=True).str.replace(r"Inc$", "Inc.", regex=True).str.replace(r"Corp$", "Corp.", regex=True)

# Identify top 10 patent assignee firms
top10_firms = df.groupby("firm_display")["patent_num"].nunique().sort_values(ascending=False).head(10).index.tolist()
df_top10 = df[df["firm_display"].isin(top10_firms)]

# Pivot counts by firm and cluster group
firm_group_counts = (
    df_top10.groupby(["firm_display", "group_short"])["patent_num"]
    .nunique()
    .unstack(fill_value=0)
)

# Sort firms so highest total patent count is at the top
firm_order = df_top10.groupby("firm_display")["patent_num"].nunique().loc[firm_group_counts.index].sort_values(ascending=True).index
firm_group_counts = firm_group_counts.loc[firm_order]

# Horizontal stacked bar plot
ax = firm_group_counts.plot(
    kind="barh",
    stacked=True,
    figsize=(11, 7),
    colormap="tab10",
    edgecolor="white",
    linewidth=0.8
)
plt.title("Top 10 Patent Assignees: Privacy Patents by Cluster Group", fontsize=14, weight="bold", pad=15)
plt.xlabel("Number of Patents", fontsize=12)
plt.ylabel("Assignee Firm", fontsize=12)
plt.legend(title="Cluster Group", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
plt.tight_layout()
plt.show()


### Bonus: Top 10 Assignees Across the 4 Specific Privacy Issues

Displays the distribution of patents for the top 10 firms across the 4 specific focal issues (Mobile Telephony, Online Ads & Web, Anonymizing & Identity, User Tracking H04L).

In [ ]:
# Aggregate by firm for each focal issue
top10_issues = pd.DataFrame({
    "1. Mobile Telephony (H04W)": df_top10[df_top10["issue_1_mobile_telephony"]].groupby("firm_display")["patent_num"].nunique(),
    "2. Online Ads, Cookies & Web": df_top10[df_top10["issue_2_online_ads_cookies"]].groupby("firm_display")["patent_num"].nunique(),
    "3. Anonymizing & Identity": df_top10[df_top10["issue_3_anonymizing_identity"]].groupby("firm_display")["patent_num"].nunique(),
    "4. Tracking User Activity (H04L)": df_top10[df_top10["issue_4_tracking_h04l"]].groupby("firm_display")["patent_num"].nunique()
}).fillna(0).astype(int).loc[firm_order]

ax = top10_issues.plot(
    kind="barh",
    stacked=True,
    figsize=(11, 7),
    colormap="Set2",
    edgecolor="white",
    linewidth=0.8
)
plt.title("Top 10 Patent Assignees: Patent Counts Across the 4 Focal Issues", fontsize=14, weight="bold", pad=15)
plt.xlabel("Number of Patents", fontsize=12)
plt.ylabel("Assignee Firm", fontsize=12)
plt.legend(title="Focal Privacy Issue", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
plt.tight_layout()
plt.show()
